# Cloud Simulations with pyRadtran

This notebook demonstrates the enhanced cloud functionality in pyRadtran, including:
- Generating cloud files from ERA5 datasets
- Parametric cloud definitions
- Using existing libRadtran cloud files
- Comparison of different cloud scenarios

In [9]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

# Import pyRadtran modules
from pyradtran.config import SimulationConfig
from pyradtran.clouds import CloudGenerator, CloudFileWriter, generate_cloud_file_from_era5

# Setup paths
config_dir = Path('../config')
work_dir = Path('./work')
work_dir.mkdir(exist_ok=True)

## 1. Creating Cloud Files from ERA5 Data

First, let's create some synthetic ERA5-like cloud data and demonstrate how to generate cloud files.

In [10]:
# Create synthetic ERA5-like dataset
pressure_levels = np.array([1000, 950, 900, 850, 800, 750, 700, 650, 600, 550, 500, 450, 400, 350, 300, 250, 200, 150, 100])
times = [datetime(2020, 4, 1, 12, 0)]
latitudes = np.arange(75, 80, 0.5)
longitudes = np.arange(-20, -10, 0.5)

# Create realistic geopotential height profile (in meters)
# Using standard atmosphere approximation: z ≈ -7000 * ln(p/p0)
geopotential_height = -7000 * np.log(pressure_levels / 1013.25)
geopotential_height = np.maximum(geopotential_height, 0)  # Ensure no negative altitudes

# Create cloud data (synthetic)
np.random.seed(42)
clwc = np.random.exponential(1e-6, (len(times), len(pressure_levels), len(latitudes), len(longitudes)))
ciwc = np.random.exponential(5e-7, (len(times), len(pressure_levels), len(latitudes), len(longitudes)))
cc = np.random.uniform(0, 1, (len(times), len(pressure_levels), len(latitudes), len(longitudes)))
temperature = 288.15 - 0.0065 * geopotential_height[:, None, None]  # Standard atmosphere temperature
temperature = np.broadcast_to(temperature, clwc.shape)

# Make clouds more realistic (more water at low levels, more ice at high levels)
for i, (p, z) in enumerate(zip(pressure_levels, geopotential_height)):
    if z < 3000:  # Low levels (< 3 km) - more water clouds
        clwc[:, i, :, :] *= 5
        ciwc[:, i, :, :] *= 0.1
    elif z > 8000:  # High levels (> 8 km) - more ice clouds
        clwc[:, i, :, :] *= 0.1
        ciwc[:, i, :, :] *= 3

# Create xarray dataset
era5_ds = xr.Dataset({
    'clwc': (['time', 'level', 'latitude', 'longitude'], clwc),
    'ciwc': (['time', 'level', 'latitude', 'longitude'], ciwc),
    'cc': (['time', 'level', 'latitude', 'longitude'], cc),
    't': (['time', 'level', 'latitude', 'longitude'], temperature),
    'z': (['level'], geopotential_height)  # Geopotential height in meters
}, coords={
    'time': times,
    'level': pressure_levels,
    'latitude': latitudes,
    'longitude': longitudes
})

print(f"Created synthetic ERA5 dataset with shape: {era5_ds.clwc.shape}")
print(f"LWC range: {era5_ds.clwc.min().values:.2e} - {era5_ds.clwc.max().values:.2e} kg/kg")
print(f"IWC range: {era5_ds.ciwc.min().values:.2e} - {era5_ds.ciwc.max().values:.2e} kg/kg")
print(f"Altitude range: {era5_ds.z.min().values/1000:.1f} - {era5_ds.z.max().values/1000:.1f} km")

Created synthetic ERA5 dataset with shape: (1, 19, 10, 20)
LWC range: 1.16e-12 - 4.09e-05 kg/kg
IWC range: 1.54e-12 - 1.07e-05 kg/kg
Altitude range: 0.1 - 16.2 km


### Important: Geopotential Height (`z`) Variable

When working with ERA5 cloud data, it's crucial to include the **geopotential height (`z`) variable** in your dataset. This provides accurate altitude information for each pressure level, which is essential for:

1. **Accurate cloud positioning**: Ensures clouds are placed at correct altitudes
2. **Layer boundary calculation**: Prevents issues with zero-thickness or inverted layers
3. **Realistic vertical structure**: Accounts for atmospheric variations

**Key Points:**
- The `z` variable should contain geopotential height in meters
- pyRadtran automatically converts this to geometric height in kilometers
- Without `z`, the system falls back to pressure-based altitude approximation (less accurate)
- Always include `'z': 'z'` in your `era5_cloud_variables` mapping

```python
era5_cloud_variables = {
    'lwc': 'clwc',    # Cloud liquid water content
    'iwc': 'ciwc',    # Cloud ice water content
    'cc': 'cc',       # Cloud cover
    'temp': 't',      # Temperature
    'z': 'z'          # Geopotential height (IMPORTANT!)
}
```

In [7]:
era5_ds

<xarray.Dataset> Size: 122kB
Dimensions:    (time: 1, level: 19, latitude: 10, longitude: 20)
Coordinates:
  * time       (time) datetime64[ns] 8B 2020-04-01T12:00:00
  * level      (level) int64 152B 1000 950 900 850 800 ... 300 250 200 150 100
  * latitude   (latitude) float64 80B 75.0 75.5 76.0 76.5 ... 78.5 79.0 79.5
  * longitude  (longitude) float64 160B -20.0 -19.5 -19.0 ... -11.5 -11.0 -10.5
Data variables:
    clwc       (time, level, latitude, longitude) float64 30kB 2.346e-06 ... ...
    ciwc       (time, level, latitude, longitude) float64 30kB 1.757e-08 ... ...
    cc         (time, level, latitude, longitude) float64 30kB 0.668 ... 0.5455
    t          (time, level, latitude, longitude) float64 30kB 287.6 ... 182.8
    z          (level) float64 152B 92.14 451.2 829.7 ... 1.337e+04 1.621e+04

In [8]:
# Generate cloud files from ERA5 data
lat_point = 77.5
lon_point = -15.0
sim_time = datetime(2020, 4, 1, 12, 0)

# Generate water cloud file
wc_file = work_dir / 'water_cloud_era5.dat'
generate_cloud_file_from_era5(
    era5_dataset=era5_ds,
    output_path=wc_file,
    cloud_type='wc',
    time=sim_time,
    lat=lat_point,
    lon=lon_point,
    lwc_threshold=1e-6,
    altitude_resolution_km=0.2
)

# Generate ice cloud file
ic_file = work_dir / 'ice_cloud_era5.dat'
generate_cloud_file_from_era5(
    era5_dataset=era5_ds,
    output_path=ic_file,
    cloud_type='ic',
    time=sim_time,
    lat=lat_point,
    lon=lon_point,
    iwc_threshold=1e-6,
    altitude_resolution_km=0.2
)

print(f"Generated cloud files:")
print(f"  Water cloud: {wc_file}")
print(f"  Ice cloud: {ic_file}")

TypeError: CloudGenerator.from_era5_dataset() got an unexpected keyword argument 'altitude_resolution_km'

In [ ]:
# Visualize the generated cloud profiles
def read_cloud_file(file_path):
    """Read libRadtran cloud file and return altitude, content, radius."""
    data = np.loadtxt(file_path, skiprows=2)
    return data[:, 0], data[:, 1], data[:, 2]

# Read cloud files
wc_alt, wc_lwc, wc_reff = read_cloud_file(wc_file)
ic_alt, ic_iwc, ic_reff = read_cloud_file(ic_file)

# Plot cloud profiles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))

# Water cloud profile
ax1.plot(wc_lwc, wc_alt, 'b-', linewidth=2, label='LWC')
ax1.set_xlabel('Liquid Water Content (g/m³)')
ax1.set_ylabel('Altitude (km)')
ax1.set_title('Water Cloud Profile (from ERA5)')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(left=0)

# Ice cloud profile
ax2.plot(ic_iwc, ic_alt, 'r-', linewidth=2, label='IWC')
ax2.set_xlabel('Ice Water Content (g/m³)')
ax2.set_ylabel('Altitude (km)')
ax2.set_title('Ice Cloud Profile (from ERA5)')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(left=0)

plt.tight_layout()
plt.show()

print(f"Water cloud layers: {np.sum(wc_lwc > 0)} / {len(wc_lwc)}")
print(f"Ice cloud layers: {np.sum(ic_iwc > 0)} / {len(ic_iwc)}")

FileNotFoundError: work/water_cloud_era5.dat not found.

## 2. Running Cloud Simulations

Now let's run radiative transfer simulations with different cloud scenarios.

In [ ]:
# Load different cloud configurations
configs = {
    'clear_sky': SimulationConfig.from_yaml(config_dir / 'default_simulation.yaml'),
    'parametric_clouds': SimulationConfig.from_yaml(config_dir / 'cloud_parametric_example.yaml'),
    'file_clouds': SimulationConfig.from_yaml(config_dir / 'cloud_file_example.yaml')
}

# Disable clouds for clear sky scenario
configs['clear_sky'].simulation_defaults.clouds.enabled = False

print("Loaded configurations:")
for name, config in configs.items():
    cloud_status = "enabled" if config.simulation_defaults.clouds.enabled else "disabled"
    print(f"  {name}: clouds {cloud_status}")

In [ ]:
# Run simulations for comparison
simulation_time = datetime(2020, 4, 1, 12, 0)
latitude = 77.5
longitude = -15.0

results = {}

for scenario_name, config in configs.items():
    print(f"Running {scenario_name} simulation...")
    
    # Create PyRadtran interface
    pyrad = PyRadtran(config)
    
    try:
        # Run single point simulation
        result = pyrad.run_single_point(
            time=simulation_time,
            latitude=latitude,
            longitude=longitude
        )
        
        if result is not None:
            results[scenario_name] = result
            print(f"  ✓ {scenario_name} completed successfully")
        else:
            print(f"  ✗ {scenario_name} failed")
            
    except Exception as e:
        print(f"  ✗ {scenario_name} error: {e}")

print(f"\nCompleted {len(results)} simulations")

In [ ]:
# Compare simulation results
if results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Colors for different scenarios
    colors = {'clear_sky': 'blue', 'parametric_clouds': 'red', 'file_clouds': 'green'}
    
    for scenario, ds in results.items():
        color = colors.get(scenario, 'black')
        
        # Plot downward irradiance
        if 'edn' in ds:
            axes[0, 0].plot(ds.edn.isel(lambda=5), ds.zout, 
                          color=color, label=scenario, linewidth=2)
    
        # Plot upward irradiance  
        if 'eup' in ds:
            axes[0, 1].plot(ds.eup.isel(lambda=5), ds.zout,
                          color=color, label=scenario, linewidth=2)
            
        # Plot direct irradiance
        if 'edir' in ds:
            axes[1, 0].plot(ds.edir.isel(lambda=5), ds.zout,
                          color=color, label=scenario, linewidth=2)
            
        # Plot global irradiance at surface
        if 'eglo' in ds:
            axes[1, 1].plot(ds.lambda_nm, ds.eglo.isel(zout=0),
                          color=color, label=scenario, linewidth=2)
    
    # Formatting
    axes[0, 0].set_xlabel('Downward Irradiance (W m⁻² nm⁻¹)')
    axes[0, 0].set_ylabel('Altitude (km)')
    axes[0, 0].set_title('Downward Irradiance Profile')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_xlabel('Upward Irradiance (W m⁻² nm⁻¹)')
    axes[0, 1].set_ylabel('Altitude (km)')
    axes[0, 1].set_title('Upward Irradiance Profile')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_xlabel('Direct Irradiance (W m⁻² nm⁻¹)')
    axes[1, 0].set_ylabel('Altitude (km)')
    axes[1, 0].set_title('Direct Irradiance Profile')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_xlabel('Wavelength (nm)')
    axes[1, 1].set_ylabel('Global Irradiance (W m⁻² nm⁻¹)')
    axes[1, 1].set_title('Surface Global Irradiance Spectrum')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print some summary statistics
    print("\nSummary Statistics (at surface):")
    for scenario, ds in results.items():
        if 'eglo' in ds:
            surface_total = ds.eglo.isel(zout=0).sum().values
            print(f"  {scenario}: Total surface irradiance = {surface_total:.2f} W m⁻²")
else:
    print("No successful simulations to compare")

## 3. Direct ERA5 Integration Example

This shows how to use ERA5 datasets directly in simulations (programmatic approach).

In [ ]:
# Example of using ERA5 dataset directly in configuration
era5_config = SimulationConfig.from_yaml(config_dir / 'cloud_era5_example.yaml')

# Set the ERA5 dataset programmatically (can't be done in YAML)
era5_config.simulation_defaults.clouds.era5_dataset = era5_ds
era5_config.simulation_defaults.clouds.era5_time = simulation_time.isoformat() + 'Z'
era5_config.simulation_defaults.clouds.era5_lat = latitude
era5_config.simulation_defaults.clouds.era5_lon = longitude

print("ERA5-integrated configuration:")
print(f"  Cloud source: {era5_config.simulation_defaults.clouds.cloud_source}")
print(f"  Cloud type: {era5_config.simulation_defaults.clouds.cloud_type}")
print(f"  ERA5 dataset shape: {era5_ds.clwc.shape}")
print(f"  Target time: {era5_config.simulation_defaults.clouds.era5_time}")
print(f"  Target location: {era5_config.simulation_defaults.clouds.era5_lat}°N, {era5_config.simulation_defaults.clouds.era5_lon}°E")

In [ ]:
# Run ERA5-based simulation
print("Running ERA5-integrated simulation...")

try:
    pyrad_era5 = PyRadtran(era5_config)
    
    era5_result = pyrad_era5.run_single_point(
        time=simulation_time,
        latitude=latitude,
        longitude=longitude
    )
    
    if era5_result is not None:
        print("✓ ERA5 simulation completed successfully")
        
        # Quick comparison with clear sky
        if 'clear_sky' in results and 'eglo' in era5_result:
            clear_surface = results['clear_sky'].eglo.isel(zout=0).sum().values
            era5_surface = era5_result.eglo.isel(zout=0).sum().values
            cloud_effect = (era5_surface - clear_surface) / clear_surface * 100
            
            print(f"\nCloud radiative effect:")
            print(f"  Clear sky: {clear_surface:.2f} W m⁻²")
            print(f"  With ERA5 clouds: {era5_surface:.2f} W m⁻²")
            print(f"  Relative change: {cloud_effect:.1f}%")
    else:
        print("✗ ERA5 simulation failed")
        
except Exception as e:
    print(f"✗ ERA5 simulation error: {e}")

## 4. Cloud Utility Functions

Demonstrate the utility functions for cloud analysis and generation.

In [ ]:
# Generate cloud layers using the utility functions
from pyradtran.clouds import CloudLayer, CloudGenerator

# Create simple parametric clouds
simple_clouds = CloudGenerator.from_simple_parameters(
    z_base_km=2.0,
    z_top_km=4.0,
    lwc_g_m3=0.2,
    r_eff_um=12.0,
    n_layers=5
)

print(f"Generated {len(simple_clouds)} simple cloud layers:")
for i, layer in enumerate(simple_clouds):
    print(f"  Layer {i+1}: {layer.z_bottom_km:.1f}-{layer.z_top_km:.1f} km, "
          f"LWC={layer.lwc_g_m3:.2f} g/m³, R_eff={layer.r_eff_um:.1f} μm")

In [ ]:
# Extract cloud layers from ERA5 dataset
era5_cloud_layers = CloudGenerator.from_era5_dataset(
    ds=era5_ds,
    time=simulation_time,
    lat=latitude,
    lon=longitude,
    lwc_threshold=1e-6,
    iwc_threshold=1e-6
)

print(f"\nExtracted {len(era5_cloud_layers)} cloud layers from ERA5:")
for i, layer in enumerate(era5_cloud_layers[:10]):  # Show first 10
    cloud_type = "Water" if layer.lwc_g_m3 > layer.iwc_g_m3 else "Ice"
    dominant_content = max(layer.lwc_g_m3, layer.iwc_g_m3)
    print(f"  Layer {i+1}: {layer.z_bottom_km:.1f}-{layer.z_top_km:.1f} km, "
          f"{cloud_type}, Content={dominant_content:.3e} g/m³")

if len(era5_cloud_layers) > 10:
    print(f"  ... and {len(era5_cloud_layers) - 10} more layers")

In [ ]:
# Write custom cloud file
custom_wc_file = work_dir / 'custom_water_cloud.dat'
CloudFileWriter.write_water_cloud_file(
    cloud_layers=simple_clouds,
    output_path=custom_wc_file,
    altitude_resolution_km=0.1
)

print(f"\nCreated custom cloud file: {custom_wc_file}")
print("\nFirst 10 lines of the file:")
with open(custom_wc_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"  {line.strip()}")
        else:
            break

## Summary

This notebook demonstrated the enhanced cloud functionality in pyRadtran:

1. **ERA5 Integration**: Automatically generate cloud files from ERA5 datasets
2. **Parametric Clouds**: Define clouds using simple layer parameters
3. **File-based Clouds**: Use existing libRadtran cloud files
4. **Comparison**: Compare radiative effects of different cloud scenarios
5. **Utilities**: Cloud generation and analysis tools

The new cloud functionality provides a flexible and powerful way to include realistic cloud effects in radiative transfer simulations, with seamless integration into the existing ERA5/xarray ecosystem.